# Exploring Agentic Rags with Llamaindex and Tavily

In [20]:
from tavily import AsyncTavilyClient
from llama_index.core.tools import FunctionTool
from llama_index.core import Settings
from llama_index.core.agent.workflow import ReActAgent, AgentInput, AgentStream, ToolCall, ToolCallResult, AgentOutput
from llama_index.core.workflow import Context
import os
from llama_index.llms.groq import Groq

In [21]:
with open('tavily_key.txt', 'r', encoding='utf-8') as f:
    TAVILY_API_KEY = f.readline().strip()

with open('groq_key.txt', 'r', encoding='utf-8') as f:
    GROQ_API_KEY = f.readline().strip()

os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

In [22]:
async def web_search(query: str) -> str:
    """Use the web to get up-to-date information and sources"""
    client = AsyncTavilyClient(api_key=TAVILY_API_KEY)
    result = await client.search(query, search_depth="advanced", include_raw_content=False)
    return str(result)

In [23]:
web_search_tool = FunctionTool.from_defaults(
    fn=web_search,
    name="web_search",
    description="Search the web for fresh information and return relevant results with links"
)

In [24]:
llm = Groq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
Settings.llm = llm
tools = [web_search_tool]

agent = ReActAgent(
    name="WebScraper",
    tools=tools,
    llm=llm,
    max_iterations=5,
    system_prompt=(
        "You are a research and planning assistant. "
        "Use tools when helpful. "
        "When using web_search tool, summarize concisely and include source links. "
        "Show brief, actionable plans. IMPORTANT: If you have enough information, stop reasoning and return the final answer."
    )
)


In [25]:
ctx = Context(agent) #store query and response history
query = (
    "I'm planning a day trip to Berkeley this Sunday. "
    "Find 2-3 must-do activities (with links), estimate total ticket costs for two adults "
    "if needed, and suggest a 6-hour itinerary. Keep it tight."
)

#Kick off workflow and return an handler
handler = agent.run(user_msg=query, ctx=ctx)

#Stream events as they happen
async for event in handler.stream_events():
    if isinstance(event, AgentInput):
        print(f"\nAgentInput from {event.current_agent_name}:\n{event.input}")
    elif isinstance(event, AgentStream):
        print(event.delta, end="", flush=True)
    elif isinstance(event, ToolCall):
        print(f"Agent requested a ToolCall: {event.tool_name} with args: {event}")
    elif isinstance(event, ToolCallResult):
        print(f"ToolResult {event.tool_name}: {str(event.tool_kwargs)[:1000]}...")
    elif isinstance(event, AgentOutput):
        print(f"\nFinal result from {event.current_agent_name}:\n{event.response.content}\n")



AgentInput from WebScraper:
[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are designed to help with a variety of tasks, from answering questions to providing summaries to other types of analyses.\n\n## Tools\n\nYou have access to a wide variety of tools. You are responsible for using the tools in any sequence you deem appropriate to complete the task at hand.\nThis may require breaking the task into subtasks and using different tools to complete each subtask.\n\nYou have access to the following tools:\n> Tool Name: web_search\nTool Description: Search the web for fresh information and return relevant results with links\nTool Args: {"properties": {"query": {"title": "Query", "type": "string"}}, "required": ["query"], "type": "object"}\n\n\n\n## Output Format\n\nPlease answer in the same language as the question and use the following format:\n\n```\nThought: The current language of the user is: (user\'s language). 